Idea: Use Imagemagick to create images that JUST contain the annotations (text and graphic), so each will be very small.
Then, we can use gimp to layer the two relevent thumb..bmp images and the annotation images. 
Then we can use gimp to shift from image to image, and calculate the difference and subtraction images.  

In [3]:
import numpy as np

In [4]:
import wand
from wand.image import Image
from wand.drawing import Drawing

In [5]:
import math

In [6]:
import matplotlib.pyplot as plt

In [7]:
dt = np.dtype([('frame', np.int32), 
               ('Rd','<i8'),('Rx',np.int16),('Ry',np.int16),
               ('Gd','<i8'),('Gx',np.int16),('Gy',np.int16),
               ('Bd','<i8'),('Bx',np.int16),('By',np.int16),
               ('rd','<i8'),('rx',np.int16),('ry',np.int16),
               ('gd','<i8'),('gx',np.int16),('gy',np.int16),
               ('bd','<i8'),('bx',np.int16),('by',np.int16),
               ('Rn',np.int32),('Gn',np.int32),('Bn',np.int32),
               ('rn',np.int32),('gn',np.int32),('bn',np.int32),
               ('stn',np.int32)]
             )

In [18]:
intdata = np.loadtxt("DroneShort1HalfDecimated.int.1.txt",converters=float,dtype=dt)
nframes=len(intdata)
IPath="/media/seth/CTAP/bitmaps-jobDS1HalfDecimatedFRAME"

In [19]:
def to6( n ):
    return "{!s:>06}".format(n)
def tothumb(n):
    return IPath+"/thumb"+to6(n)+".bmp"
tothumb(1234)

'/media/seth/CTAP/bitmaps-jobDS1HalfDecimatedFRAME/thumb001234.bmp'

In [20]:
width=0
height=0
def setwh(n):
    img=Image(filename=tothumb(1))
    global width
    width=img.width
    global height
    height=img.height
setwh(1)
widthTo1920=math.ceil(float(width)/1920.0)

In [21]:
bgimg=Image(width=width, height=height)

In [22]:
bgimg.save(filename="trybg.png")

Setup for drawing the data line.

In [23]:
llcapx=int(width/20)
llcapy=int(height-width/20)

In [34]:
drdbl=Drawing()
drgbl=drdbl.clone()
drdbl.font_size=30*widthTo1920
drdbl.fill_color="ORANGE" #Instead of WHITE so it shows in Jupyter where transparent background is shown in white.
drdbl.stroke_color="ORANGE"

Setup for drawing markers on Phase1a selected pixels.

In [35]:
def M(TH) :
    return( np.array( [ [math.cos(math.pi*TH/180.), math.sin(math.pi*TH/180.)], [-math.sin(math.pi*TH/180.), math.cos(math.pi*TH/180.)] ] ) )

#Geometry of normalized unit arrows to show RGB changes
TH=30.0                  #angle of arrows away from vertical, and hands away from body
lcircr=0.1               #little circle radius
hslen=0.1                #length of each hand of an
# Unit Vectors
g1=np.array([0.0,1.0])   #unit lower case, down, green

#tiny vectors
tc=lcircr*np.array([0.0,1.0]) #radius (down, y dir of tiny circle, and foot of down unit arrow
def T(s) :
    return (tc + g1*s) #tail on tiny circle, head down by unit * s (scale, 0<=s<=1)

#Arrow body is (tc->T(s)*M..
def arrB(s) :
    return np.array([tc, T(s)])

TL=hslen*g1@M(180.0+TH) #coord of left hand rel to head
TR=hslen*g1@M(180.0-TH) #coord of right hand rel to head

#down dir Left arm LA(s) is (T(s)->TL(s))
def arrL(s) :
    return np.array([T(s), T(s)+TL])


#down dir Right arm LA(s) is (T(s)->TL(s))
def arrR(s) :
    return np.array([T(s), T(s)+TR])

def unit_up_arrow(s) :
    return -np.concat([arrB(s),arrL(s),arrR(s)])


In [36]:
def dispdata(row):
    #print(fn, row['frame'])
    return ( [ row['Rd'],   -30.0, [row['Rx'],row['Ry']], row['Rn' ], "red" ],
             [ row['Gd'],     0.0, [row['Gx'],row['Gy']], row['Gn' ], "green" ],
             [ row['Bd'],    30.0, [row['Bx'],row['By']], row['Bn' ], "blue" ],
             [ -row['rd'], -150.0, [row['rx'],row['ry']], row['rn' ], "red" ],
             [ -row['gd'],  180.0, [row['gx'],row['gy']], row['gn' ], "green" ],
             [ -row['bd'],  150.0, [row['rx'],row['by']], row['bn' ], "blue" ] )             

In [37]:
colvaldiv = float(128)
arrlen = 100*widthTo1920

def drdata(dwg, row):
    data = dispdata(row)
    for r in data:
        #print(r)
        #print((r[0]/colvaldiv))
        #print( "arrow", arrlen*unit_up_arrow( (r[0]/colvaldiv)) )
        line = np.round( (arrlen*unit_up_arrow(r[0]/colvaldiv))@M(r[1]) + r[2]).astype(int)
        dwg.stroke_width = 2
        dwg.stroke_color = wand.color.Color( r[4] )
        for i in range(0,3):
            dwg.line(line[2*i],line[2*i+1])

In [38]:
img=0
def vis(fn):
    global img
    img=Image(filename=tothumb(fn))
    row=np.array(intdata[fn-1])
    t=intdata[fn-1].item(0)
    intdatarowstr = str(t[0])
    for z in range(1,24,3):
        intdatarowstr+=" "+str(t[z:z+3])
    intdatarowstr+="  "+str(t[24])
    global drdbl
    draw=drdbl.clone()
    draw.text(llcapx,llcapy,intdatarowstr)
    drdata(draw, row )
    draw(img)
    #draw
    #draw(img)
    return img #so jupyter tries to print the result which makes the picture appear!      

In [39]:
def visonlydata(fn):
    img=bgimg.clone()
    row=np.array(intdata[fn-1])
    t=intdata[fn-1].item(0)
    intdatarowstr = str(t[0])
    for z in range(1,24,3):
        intdatarowstr+=" "+str(t[z:z+3])
    intdatarowstr+="  "+str(t[24])
    draw=drdbl.clone()
    draw.text(llcapx,llcapy,intdatarowstr)
    drdata(draw, row )
    draw(img)
    #draw
    #draw(img)
    return img #so jupyter tries to print the result which makes the picture appear!      

In [43]:
#imggot=visonlydata(319)

In [44]:
#imggot.save(filename="imggot.png")

In [45]:
#imggot

In [64]:
imggot.merge_layers?

Signature: imggot.merge_layers(method)
Docstring:
Composes all the image layers from the current given image onward
to produce a single image of the merged layers.

The initial canvas's size depends on the given ImageLayerMethod, and is
initialized using the first images background color.  The images
are then composited onto that image in sequence using the given
composition that has been assigned to each individual image.
The method must be set with a value from :const:`IMAGE_LAYER_METHOD`
that is acceptable to this operation. (See ImageMagick documentation
for more details.)

:param method: the method of selecting the size of the initial canvas.
:type method: :class:`str`

.. versionadded:: 0.4.3
File:      /data/Software/sklearn-venv/lib/python3.10/site-packages/wand/image.py
Type:      method

In [119]:
def newvis(fn):
    skyimg=Image(filename=tothumb(fn))
    dataimg=visonlydata(fn)
    draw=Drawing()
    draw.composite(image=skyimg,operator="overlay",left=0,top=0,width=width,height=height)
    draw(dataimg)
    return dataimg

In [120]:
#visonlydata(319)

In [133]:
resultimg=newvis(319)

In [136]:
#resultimg

In [135]:
resultbmp=resultimg.clone()
resultbmp.save(filename="resultbmp.bmp")
resultimg.compression_quality=100
resultimg.format='jpeg'
resultimg.save(filename="resultjpeg.jpeg")
#resulting.

In [132]:
resultimg.save(filename="myresult.jpeg")

In [19]:
#comment out so we dont have a huge .ipnb file
#vis(319)

In [17]:
def tophonea(n):
    return IPath+"/phonea"+to6(n)+".jpg"
def doAll():
    for i in range(nframes):
        fn=i+1
        img=vis(fn)
        print("visualized frame", fn, end="")
        img.format = 'jpeg'
        img.save(filename=tophonea(fn))
        print(" saved", fn, end="\r")

In [18]:
doAll()